In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import tokenize
import pdb
import os
import math
from datasets import load_dataset
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [ ]:
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

In [ ]:
train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transforms.ToTensor())

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

In [ ]:
class DawidGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.embeddings = nn.Linear(768, 1024)
        self.attention_blocks = nn.ModuleList([SelfAttention() for _ in range(4)])
        self.ffn_blocks = nn.ModuleList([FeedForwardNet() for _ in range(4)])
        self.output_layer = nn.Linear(1024, len(train_dataset))
        self.norm1 = nn.LayerNorm(1024)
        self.norm2 = nn.LayerNorm(1024)
        #frequency = self.calculate_frequencies()
        #positional_encodings = self.calculate_positional_encodings(frequency)
        #self.register_buffer('positional_encodings', positional_encodings)
        self.positional_encodings = torch.rand((4, 1024))
    
    def calculate_positional_encodings(self, frequency):
        position_vector = torch.reshape(torch.arange(64), (64, 1)).float()
        frequency = torch.reshape(frequency, (1, 64))
        function_input = torch.matmul(position_vector, frequency)
        sin_values = torch.sin(function_input)
        cos_values = torch.cos(function_input)
        positional_encodings = torch.stack((sin_values, cos_values), dim=2).reshape(8, 1024)
        return positional_encodings
    
    def calculate_frequencies(self):
        pair_indices = torch.arange(64) * 2
        exponent = pair_indices / 1024
        denominator = 10000 ** exponent
        frequency = 1 / denominator
        return frequency
    
    def forward(self, x):
        x = self.embeddings(x)
        x = x + self.positional_encodings
        
        for i in range(4):
            residual = x
            x = self.norm1(x)
            attn_out = self.attention_blocks[i](x)
            x = residual + 0.1 * attn_out

            residual = x 
            x = self.norm2(x)
            ffn_out = self.ffn_blocks[i](x)
            x = residual + 0.1 * ffn_out
        
        x = self.output_layer(x)

        return x

In [ ]:
class SelfAttention(nn.Module):
        def __init__(self):
            super(SelfAttention, self).__init__()
            self.query_weights = nn.Parameter(torch.rand(1024, 1024))
            self.key_weights = nn.Parameter(torch.rand(1024, 1024))
            self.value_weights = nn.Parameter(torch.rand(1024, 1024))
    
        def calculate_parameters(self):
            self.query = torch.matmul(self.input_vec, self.query_weights)
            self.key = torch.transpose(torch.matmul(self.input_vec, self.key_weights), 1, 2)
            self.value = torch.matmul(self.input_vec, self.value_weights)
    
        def multi_head_attention(self):
            batch_size = self.query.shape[0]
            seq_len = self.query.shape[1]
            self.query = torch.reshape(self.query, (batch_size, seq_len, 4, 1024))
            self.query = torch.transpose(self.query, 1, 2)
            self.key = torch.reshape(self.key, (batch_size, seq_len, 4, 1024))
            self.key = torch.transpose(self.key, 1, 2)
            self.key = torch.transpose(self.key, 2, 3)
            self.value = torch.reshape(self.value, (batch_size, seq_len, 4, 1024))
            self.value = torch.transpose(self.value, 1, 2)
                
        def calculate_attention_score(self):
            self.attention_scores = torch.matmul(self.query, self.key).float()  / math.sqrt(128)
            
        def normalize_softmax(self):
            self.normalized_values = torch.nn.functional.softmax(self.attention_scores, dim=1)
    
        def create_representation(self):
            self.representations = torch.matmul(self.normalized_values, self.value)
            self.representations = torch.transpose(self.representations, 1, 2)
            batch_size = self.representations.shape[0]
            seq_len = self.representations.shape[1]  # Fixed! seq_len is at position 1 after transpose
            self.representations = torch.reshape(self.representations, (batch_size, seq_len, 4,  1024))
            
        def forward(self, x):
            self.input_vec = x
            self.calculate_parameters()
            self.multi_head_attention()
            self.calculate_attention_score()
            self.normalize_softmax()
            self.create_representation()
            return self.representations

In [ ]:
 class FeedForwardNet(nn.Module):
        def __init__(self):
            super(FeedForwardNet, self).__init__()
            self.fc1 = nn.Linear(1024, 2048)
            self.fc2 = nn.Linear(2048, 1024)
    
        def forward(self, x):
            x = self.fc1(x)
            x = F.relu(x)
            output = self.fc2(x)
            return output

In [ ]:
def create_patches(permuted_shapes, resolution_P):
    shape_of_image = permuted_shapes[0].shape
    sequence_len_n = shape_of_image[0] * shape_of_image[1] // resolution_P ** 2
    P_C = (resolution_P ** 2) * shape_of_image[2]
    patches_list = [permuted_shapes[i].reshape((N, P_C)) for i in range(len(permuted_shapes))]
    return torch.stack(patches_list)

In [ ]:
def create_slice_batches(patches_list):
    start_pos = torch.randint(0, 49934, (32, )).reshape(32, 1)
    addition_value = torch.arange(0, 65)
    indicies = start_pos + addition_value
    stacked_batches = patches_list[indicies]
    input_batches = stacked_batches[:, :64]
    target_batches = stacked_batches[:, 1:],
    return input_batches, target_batches